In [1]:
# run once in Colab
!pip install -q openai google-generativeai pillow tqdm requests pandas datasets openpyxl pyarrow

In [2]:
# Cell 1
import os
import io
import time
import json
import base64
import requests
from tqdm import tqdm
from PIL import Image
import pandas as pd
from datasets import load_from_disk, Dataset, Features, Value
import traceback

In [8]:
!unzip "/content/batch_prompts_1000_WITH_IMAGES.zip"

Archive:  /content/batch_prompts_1000_WITH_IMAGES.zip
   creating: content/batch_prompts_1000_WITH_IMAGES/
  inflating: content/batch_prompts_1000_WITH_IMAGES/data-00000-of-00001.arrow  
  inflating: content/batch_prompts_1000_WITH_IMAGES/dataset_info.json  
  inflating: content/batch_prompts_1000_WITH_IMAGES/state.json  


In [ ]:
from datasets import load_from_disk
ds = load_from_disk("/content/content/batch_prompts_1000_dataset")
print("Loaded prompts:", len(ds))
ds[0]

Loaded prompts: 1000


{'index': 0,
 'file_name': '000000561009.jpg',
 'prompt_type': 'implicit',
 'generated_prompt': 'Describe the image.',
 'ground_truth': 'A bird perched on top of a tree branch.',
 'thumbnail_b64': '/9j/4AAQSkZJRgABAQAAAQABAAD//gAMQXBwbGVNYXJrCv/bAEMACAYGBwYFCAcHBwkJCAoMFA0MCwsMGRITDxQdGh8eHRocHCAkLicgIiwjHBwoNyksMDE0NDQfJzk9ODI8LjM0Mv/bAEMBCQkJDAsMGA0NGDIhHCEyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMjIyMv/AABEIAKAA8AMBIgACEQEDEQH/xAAfAAABBQEBAQEBAQAAAAAAAAAAAQIDBAUGBwgJCgv/xAC1EAACAQMDAgQDBQUEBAAAAX0BAgMABBEFEiExQQYTUWEHInEUMoGRoQgjQrHBFVLR8CQzYnKCCQoWFxgZGiUmJygpKjQ1Njc4OTpDREVGR0hJSlNUVVZXWFlaY2RlZmdoaWpzdHV2d3h5eoOEhYaHiImKkpOUlZaXmJmaoqOkpaanqKmqsrO0tba3uLm6wsPExcbHyMnK0tPU1dbX2Nna4eLj5OXm5+jp6vHy8/T19vf4+fr/xAAfAQADAQEBAQEBAQEBAAAAAAAAAQIDBAUGBwgJCgv/xAC1EQACAQIEBAMEBwUEBAABAncAAQIDEQQFITEGEkFRB2FxEyIygQgUQpGhscEJIzNS8BVictEKFiQ04SXxFxgZGiYnKCkqNTY3ODk6Q0RFRkdISUpTVFVWV1hZWmNkZWZnaGlqc3R1dnd4eXqCg4SFhoeIiYqSk5SVlpeYmZqio6Slpqeoqaqys7S1tre4ubrCw8TFxsfIycrS09TV1t

In [4]:
!unzip "/content/image_stage1.zip"

Archive:  /content/image_stage1.zip
   creating: content/image_stage1/
  inflating: content/image_stage1/dataset_info.json  
  inflating: content/image_stage1/state.json  
  inflating: content/image_stage1/data-00000-of-00001.arrow  


In [9]:
from datasets import load_from_disk

stage1 = load_from_disk("/content/content/image_stage1")
print("Stage-1 rows:", len(stage1))
print(stage1[0].keys())

Stage-1 rows: 200
dict_keys(['question_id', 'image', 'question', 'answer', 'id', 'license', 'file_name', 'coco_url', 'height', 'width', 'date_captured', 'implicit_prompt', 'explicit_prompt', 'cot_prompt', 'ground_truth'])


In [13]:
from datasets import Dataset, Features, Value
import pandas as pd
import io

# Convert stage1 (with PIL) to pandas
df1 = pd.DataFrame(stage1)
ds_df = pd.DataFrame(ds)   # Notebook4 dataset

# Function: PIL Image -> bytes
def pil_to_bytes(pil_img):
    buf = io.BytesIO()
    pil_img.save(buf, format="JPEG")
    return buf.getvalue()

df1["image"] = df1["image"].apply(lambda img: pil_to_bytes(img))

# Merge using file_name
df_merged = ds_df.merge(
    df1[["file_name", "image"]],
    on="file_name",
    how="left"
)

# 🔥 FIX: Drop any unwanted columns
if "image_bytes" in df_merged.columns:
    df_merged = df_merged.drop(columns=["image_bytes"])

print(df_merged.head())
print("Merged rows:", len(df_merged))

# Define correct features
features = Features({
    "index": Value("int32"),
    "file_name": Value("string"),
    "prompt_type": Value("string"),
    "generated_prompt": Value("string"),
    "ground_truth": Value("string"),
    "thumbnail_b64": Value("string"),
    "image": Value("binary"),   # correct
})

# Create HF dataset
ds_final = Dataset.from_pandas(df_merged, features=features, preserve_index=False)
ds_final.save_to_disk("content/content/batch_prompts_1000_WITH_IMAGES")

print("SAVED FIXED dataset with REAL IMAGES!")


   index         file_name      prompt_type  \
0      0  000000561009.jpg         implicit   
1      0  000000561009.jpg         explicit   
2      0  000000561009.jpg     cot_template   
3      0  000000561009.jpg  autocot_caption   
4      0  000000561009.jpg    autocot_image   

                                    generated_prompt  \
0                                Describe the image.   
1  Describe all objects, colors, actions, and spa...   
2  Think step-by-step about the image:\n1. Identi...   
3  Here's the chain-of-thought reasoning for the ...   
4  The image shows a bird perched on a branch. Th...   

                              ground_truth  \
0  A bird perched on top of a tree branch.   
1  A bird perched on top of a tree branch.   
2  A bird perched on top of a tree branch.   
3  A bird perched on top of a tree branch.   
4  A bird perched on top of a tree branch.   

                                       thumbnail_b64  \
0  /9j/4AAQSkZJRgABAQAAAQABAAD//gAMQXBwbGVNYXJr

CastError: Couldn't cast
index: int64
file_name: string
prompt_type: string
generated_prompt: string
ground_truth: string
thumbnail_b64: string
image_x: binary
image_y: binary
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1028
to
{'index': Value('int32'), 'file_name': Value('string'), 'prompt_type': Value('string'), 'generated_prompt': Value('string'), 'ground_truth': Value('string'), 'thumbnail_b64': Value('string'), 'image': Value('binary')}
because column names don't match

In [12]:
DATASET_PATH = "/content/content/batch_prompts_1000_WITH_IMAGES"
ds = load_from_disk(DATASET_PATH)
print("Loaded rows:", len(ds))

Loaded rows: 1000


In [ ]:
DATASET_PATH = "/content/batch_prompts_1000_WITH_IMAGES"
ds = load_from_disk(DATASET_PATH)
print("Loaded rows:", len(ds))

Loaded rows: 1000


In [ ]:
!zip -r batch_prompts_1000_WITH_IMAGES.zip /content/batch_prompts_1000_WITH_IMAGES

  adding: content/batch_prompts_1000_WITH_IMAGES/ (stored 0%)
  adding: content/batch_prompts_1000_WITH_IMAGES/data-00000-of-00001.arrow (deflated 22%)
  adding: content/batch_prompts_1000_WITH_IMAGES/dataset_info.json (deflated 72%)
  adding: content/batch_prompts_1000_WITH_IMAGES/state.json (deflated 38%)


In [59]:
# Cell 2
DATASET_PATH = "/content/batch_prompts_1000_WITH_IMAGES"   # <- change if dataset path differs
OUT_DIR = "/content/notebook5_outputs"
CHECKPOINT_DIR = "/content/notebook5_checkpoints"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# How often to checkpoint (rows)
CHECKPOINT_EVERY = 50

# How many retries for an inference call
MAX_RETRIES = 6


In [69]:
!rm -r /content/notebook5_outputs
!rm -r /content/notebook5_checkpoints

In [60]:
# Cell 3
ds = load_from_disk(DATASET_PATH)
print("Loaded dataset rows:", len(ds))

# Ensure expected columns - dataset should contain at least:
# 'index','file_name','prompt_type','generated_prompt','ground_truth','image' (binary bytes)
required_cols = {"index","file_name","prompt_type","generated_prompt","ground_truth","image"}
present = set(ds.column_names)
print("Columns present:", present)
missing = required_cols - present
if missing:
    raise RuntimeError(f"Dataset missing required columns: {missing}")

# show sample
print("\nSample row keys:", ds[0].keys())

Loaded dataset rows: 1000
Columns present: {'thumbnail_b64', 'generated_prompt', 'image', 'ground_truth', 'prompt_type', 'index', 'file_name'}

Sample row keys: dict_keys(['index', 'file_name', 'prompt_type', 'generated_prompt', 'ground_truth', 'thumbnail_b64', 'image'])


In [14]:
!pip install -q google-generativeai pillow tqdm requests pandas datasets openpyxl pyarrow

In [15]:
!pip install openai

In [25]:
from openai import OpenAI
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="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"
)

In [26]:
import requests

resp = requests.get("https://openrouter.ai/api/v1/models")
models = resp.json()["data"]

for m in models:
    print(m["id"])

tngtech/tng-r1t-chimera:free
tngtech/tng-r1t-chimera
anthropic/claude-opus-4.5
openrouter/bert-nebulon-alpha
allenai/olmo-3-32b-think
allenai/olmo-3-7b-instruct
allenai/olmo-3-7b-think
google/gemini-3-pro-image-preview
x-ai/grok-4.1-fast:free
google/gemini-3-pro-preview
deepcogito/cogito-v2.1-671b
openai/gpt-5.1
openai/gpt-5.1-chat
openai/gpt-5.1-codex
openai/gpt-5.1-codex-mini
kwaipilot/kat-coder-pro:free
moonshotai/kimi-linear-48b-a3b-instruct
moonshotai/kimi-k2-thinking
amazon/nova-premier-v1
perplexity/sonar-pro-search
mistralai/voxtral-small-24b-2507
openai/gpt-oss-safeguard-20b
nvidia/nemotron-nano-12b-v2-vl:free
nvidia/nemotron-nano-12b-v2-vl
minimax/minimax-m2
liquid/lfm2-8b-a1b
liquid/lfm-2.2-6b
ibm-granite/granite-4.0-h-micro
deepcogito/cogito-v2-preview-llama-405b
openai/gpt-5-image-mini
anthropic/claude-haiku-4.5
qwen/qwen3-vl-8b-thinking
qwen/qwen3-vl-8b-instruct
openai/gpt-5-image
openai/o3-deep-research
openai/o4-mini-deep-research
nvidia/llama-3.3-nemotron-super-49b-v1.

In [122]:
for m in models:
    name = m["id"].lower()
    if "vision" in name or "vl" in name or "flash" in name or "llava" in name:
        print(name)

nvidia/nemotron-nano-12b-v2-vl:free
nvidia/nemotron-nano-12b-v2-vl
qwen/qwen3-vl-8b-thinking
qwen/qwen3-vl-8b-instruct
google/gemini-2.5-flash-image
qwen/qwen3-vl-30b-a3b-thinking
qwen/qwen3-vl-30b-a3b-instruct
google/gemini-2.5-flash-preview-09-2025
google/gemini-2.5-flash-lite-preview-09-2025
qwen/qwen3-vl-235b-a22b-thinking
qwen/qwen3-vl-235b-a22b-instruct
qwen/qwen3-coder-flash
opengvlab/internvl3-78b
meituan/longcat-flash-chat:free
meituan/longcat-flash-chat
google/gemini-2.5-flash-image-preview
baidu/ernie-4.5-vl-28b-a3b
google/gemini-2.5-flash-lite
baidu/ernie-4.5-vl-424b-a47b
google/gemini-2.5-flash
qwen/qwen2.5-vl-32b-instruct:free
qwen/qwen2.5-vl-32b-instruct
google/gemini-2.0-flash-lite-001
google/gemini-2.0-flash-001
qwen/qwen-vl-plus
qwen/qwen-vl-max
qwen/qwen2.5-vl-72b-instruct
google/gemini-2.0-flash-exp:free
meta-llama/llama-3.2-90b-vision-instruct
meta-llama/llama-3.2-11b-vision-instruct
qwen/qwen-2.5-vl-7b-instruct


In [126]:
sample = ds[0]
prompt = sample["generated_prompt"]
img_bytes = sample["image"]

print("\n=== Nemotron VL ===")
print(run_openrouter_model("nvidia/nemotron-nano-12b-v2-vl:free", prompt, img_bytes))

print("\n=== Qwen3-VL 8B ===")
print(run_openrouter_model("qwen/qwen3-vl-8b-instruct", prompt, img_bytes))

print("\n=== Gemini Flash Vision ===")
print(run_openrouter_model("google/gemini-2.5-flash-image", prompt, img_bytes))



=== Nemotron VL ===
The image captures a serene moment in nature, featuring a single bird perched delicately on a slender tree branch. The bird, positioned centrally and facing toward the right, boasts a striking contrast in its plumage: its body and wings are cloaked in rich dark brown feathers, while its head transitions to a softer, lighter brown. Its sharp, blue-gray beak stands out against the warm tones of its feathers, and its eyes—dark with vivid yellow irises—add a captivating focal point to its composed expression.  

The bird is framed against a lush, verdant backdrop, where blurred greenery, twigs, and hints of foliage create a dreamy bokeh effect. The branches surrounding it are slender and unpainted, with small buds scattered along their lengths, suggesting the natural, untamed environment of a forest or dense woodland. The composition is bathed in gentle, diffused sunlight, which enhances the tranquility of the scene. There are no texts, human-made objects, or additiona

In [93]:
import os
import base64
import json
import requests
import io
from PIL import Image
import pandas as pd
from tqdm import tqdm
from datasets import Dataset, Features, Value

In [101]:
!pip install requests pillow datasets tqdm
!pip install -q huggingface_hub

In [18]:
DATASET_PATH = "/content/batch_prompts_1000_WITH_IMAGES"
OUT_DIR = "/content/notebook5_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

ds = load_from_disk(DATASET_PATH)
print("Loaded:", len(ds))
print(ds[0])

FileNotFoundError: Directory /content/batch_prompts_1000_WITH_IMAGES not found

In [19]:
import base64, time

def run_openrouter_model(model_id, prompt, image_bytes):
    """Unified inference for ALL OpenRouter vision models"""
    img_b64 = base64.b64encode(image_bytes).decode()

    for attempt in range(5):
        try:
            resp = client.chat.completions.create(
                model=model_id,
                messages=[
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {
                                "type": "image_url",
                                "image_url": f"data:image/jpeg;base64,{img_b64}"
                            }
                        ]
                    }
                ]
            )

            return resp.choices[0].message.content

        except Exception as e:
            if "rate-limit" in str(e).lower() or "429" in str(e):
                print(f"[{model_id}] Rate limited, retrying … attempt {attempt+1}/5")
                time.sleep(2)
                continue

            return f"[ERROR] {e}"

    return f"[ERROR] Too many retries for {model_id}"


In [20]:
import io, base64
from PIL import Image

def load_image_from_bytes(b):
    return Image.open(io.BytesIO(b)).convert("RGB")

def pil_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="JPEG")
    return base64.b64encode(buf.getvalue()).decode()

def safe_extract(js):
    if isinstance(js, dict):
        for k in ("output_text","generated_text","text","result"):
            if k in js: return js[k]
        if "candidates" in js:
            return js["candidates"][0]["content"]["parts"][0]["text"]
        if "results" in js:
            return js["results"][0]["text"]
    return str(js)


In [22]:
VISION_MODELS = {
    #"nemotron_vl": "nvidia/nemotron-nano-12b-v2-vl:free",
    "qwen3_vl": "qwen/qwen3-vl-8b-instruct",
    #"gemini_flash": "google/gemini-2.5-flash-image"
}

VISION_MODELS

{'qwen3_vl': 'qwen/qwen3-vl-8b-instruct'}

In [130]:
import pandas as pd
from tqdm import tqdm
import os

OUT_DIR = "/content/notebook5_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

for model_name, model_id in VISION_MODELS.items():
    print(f"\n=== Running model: {model_name} ===")
    rows = []

    for row in tqdm(ds, desc=f"{model_name} inference"):
        prompt = row["generated_prompt"]
        img_bytes = row["image"]

        output = run_openrouter_model(model_id, prompt, img_bytes)

        rows.append({
            "index": row["index"],
            "file_name": row["file_name"],
            "prompt_type": row["prompt_type"],
            "prompt": prompt,
            "ground_truth": row["ground_truth"],
            "model_output": output
        })

    df = pd.DataFrame(rows)
    path = f"{OUT_DIR}/{model_name}_results.xlsx"
    df.to_excel(path, index=False)
    print(f"Saved → {path}")



=== Running model: nemotron_vl ===


nemotron_vl inference: 100%|██████████| 1000/1000 [3:12:51<00:00, 11.57s/it]


Saved → /content/notebook5_outputs/nemotron_vl_results.xlsx

=== Running model: qwen3_vl ===


qwen3_vl inference: 100%|██████████| 1000/1000 [1:12:55<00:00,  4.38s/it]


Saved → /content/notebook5_outputs/qwen3_vl_results.xlsx

=== Running model: gemini_flash ===


gemini_flash inference: 100%|██████████| 1000/1000 [01:17<00:00, 12.91it/s]

Saved → /content/notebook5_outputs/gemini_flash_results.xlsx


In [27]:
MODEL_NAME = "qwen3_vl"
MODEL_ID = "qwen/qwen3-vl-8b-instruct"

In [29]:
import pandas as pd
from tqdm import tqdm
import os

# pick ONE model here
MODEL_NAME = "qwen_vl_plus"
MODEL_ID = "qwen/qwen-vl-plus"

OUT_DIR = f"/content/notebook5_outputs_single/{MODEL_NAME}"
os.makedirs(OUT_DIR, exist_ok=True)

print(f"🚀 Running SINGLE MODEL INFERENCE")
print(f"Model: {MODEL_NAME} ({MODEL_ID})")
print(f"Total samples: {len(ds)}")

rows = []

for row in tqdm(ds, desc=f"{MODEL_NAME} inference"):
    prompt = row["generated_prompt"]
    img_bytes = row["image"]

    # run vision model
    output = run_openrouter_model(MODEL_ID, prompt, img_bytes)

    # store
    rows.append({
        "index": row["index"],
        "file_name": row["file_name"],
        "prompt_type": row["prompt_type"],
        "prompt": prompt,
        "ground_truth": row["ground_truth"],
        "model_output": output
    })

# save
df = pd.DataFrame(rows)
path = f"{OUT_DIR}/{MODEL_NAME}_results.xlsx"
df.to_excel(path, index=False)

print(f"✔ DONE. Saved → {path}")

🚀 Running SINGLE MODEL INFERENCE
Model: qwen_vl_plus (qwen/qwen-vl-plus)
Total samples: 1000


qwen_vl_plus inference: 100%|██████████| 1000/1000 [00:35<00:00, 28.01it/s]


✔ DONE. Saved → /content/notebook5_outputs_single/qwen_vl_plus/qwen_vl_plus_results.xlsx


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh
!ollama serve &

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Couldn't find '/root/.ollama/id_ed25519'. Generating new private key.
Your new public key is: 

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAIP2+WuQ41CkmAc9GA2so2+C5uCNbgx+SOy/560zFANqB

time=2025-11-26T22:02:23.446Z level=INFO source=routes.go:1544 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:4096 OLLAMA_DEBUG:INFO OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_C

In [ ]:
!ollama pull internvl:1b